随着对话的进行，历史消息不断累积，**state会持续增长**，为模型带来挑战：
1. LLM的**上下文窗口是有限的**，完整历史可能无法装入LLM的上下文窗口，导致上下文丢失或错误。
2. 即便模型的上下文窗口够大，多数LLM在长上下文场景仍然表现不佳。模型会**被陈旧或离题的内容“分散注意力”**。
3. 同时，会带来**高昂的token花费**。

此时需要对上下文进行管理：对历史记录进行压缩、清理、重组等。

# 消息裁剪

调用模型前裁剪上下文。

目标是控制token用量，通常**保留系统初始消息和最近若干消息**，或**按token数保留末尾内容**。

适合成本敏感、对旧上下文依赖不强的场景。

与删除的区别是，裁剪是在调用模型前进行的，Agent状态不会改变，只改变模型看见的上下文。

删除则是直接从Agent状态中移除旧消息，改变Agent状态。通常发生在模型调用之后，比如固定消息列表只能保留最近若干条消息。



# RemoveMessage到底干了什么?
当你在中间件里返回 `[RemoveMessage(id=m.id)]` 时，你实际上是向框架发送了一个**删除指令**。

框架的底层处理逻辑如下：

```
[历史消息池（内存中持续存在）]
├─ Message(id="1", content="你好，我是老王")
├─ Message(id="2", content="...")
└─ RemoveMessage(id="1")  <-- 这是一个新追加进去的“墓碑”标记
```

1. **追加“墓碑”标记**：框架收到 `RemoveMessage(id="1")` 后，并不会去内存的数组里把 id="1" 的对象删掉，而是把这个 `RemoveMessage` 作为一条新记录追加到当前线程的状态历史中。这个 `RemoveMessage` 就像是一个“墓碑”。

2. **运行时过滤合并（Reducer）**：当下一次你再次调用 `agent.invoke` 或者大模型要去读取上下文时，框架的内置合并器（Reducer）会把“原始消息”和“墓碑标记”放在一起进行计算：
> 原始消息 (id: 1) ＋ 墓碑标记 (id: 1) = ∅（对外隐藏）

它在丢给大模型之前，会自动把被标记删除的消息过滤掉。

In [1]:
# 自定义裁剪策略中间件

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)


from langchain_core.messages import HumanMessage
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any


@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    messages = state["messages"]

    if len(messages) <= 3:
        return None

    first_msg = messages[0]
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    new_messages = [first_msg] + recent_messages

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }


agent = create_agent(
    model=model,
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": [HumanMessage("你好，我是老王")]}, config)
agent.invoke({"messages": [HumanMessage("从现在起，你叫小王")]}, config)
agent.invoke({"messages": [HumanMessage("今天天气不错")]}, config)
final_response = agent.invoke({"messages": [HumanMessage("告诉我，你是谁？我是谁？")]}, config)

for msg in final_response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

好的，从现在起我叫小王。有什么我可以帮你？
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

是啊，天气不错的时候心情也容易变好。你打算出去走走，还是在家里放松一下？
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

我是一个 AI 助手，你可以把我当作你的智能对话伙伴。

你是老王。
